In [1]:
import os
import uuid
import pickle

import pandas as pd

import mlflow

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from mlflow.tracking import MlflowClient

In [2]:
year = 2021
month = 3
taxi_type = 'green'

input_file = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet'

output_file = f'output/{taxi_type}/{year:04d}-{month:02d}.parquet'

mlflow.set_tracking_uri("http://127.0.0.1:5000")

RUN_ID = os.getenv('RUN_ID', '5e1a8d1da960432b8a921ffdec3965f7')

In [3]:
def generate_uuids(n):
    return [str(uuid.uuid4()) for _ in range(n)]


def read_dataframe(filename: str):
    df = pd.read_parquet(filename)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    df['ride_id'] = generate_uuids(len(df))
    return df


def prepare_dictionaries(df: pd.DataFrame):
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    dicts = df[['PU_DO', 'trip_distance']].to_dict(orient='records')
    return dicts

In [4]:
def load_model(run_id):
    model_uri = f'runs:/{run_id}/model'
    model = mlflow.pyfunc.load_model(model_uri)
    return model


def apply_model(input_file, run_id, output_file):
    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)
    model = load_model(run_id)
    y_pred = model.predict(dicts)

    df_result = pd.DataFrame({
        'ride_id': df['ride_id'],
        'lpep_pickup_datetime': df['lpep_pickup_datetime'],
        'PULocationID': df['PULocationID'],
        'DOLocationID': df['DOLocationID'],
        'actual_duration': df['duration'],
        'predicted_duration': y_pred,
        'diff': df['duration'] - y_pred,
        'model_version': run_id
    })

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_result.to_parquet(output_file, index=False)
    print(f"✅ Predictions saved to: {output_file}")

In [5]:
apply_model(input_file=input_file, run_id=RUN_ID, output_file=output_file)


✅ Predictions saved to: output/green/2021-03.parquet


In [6]:
!ls output/green/

2021-02.parquet  2021-03.parquet
